In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Combining review text**

In [5]:
df_raw = pd.read_csv('/content/drive/MyDrive/Datasets/dataset_g2-explorer_2026-03-14_14-39-55-993.csv')

print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())

answer_cols = [c for c in df_raw.columns if c.startswith('answers/')]
print("Answer columns found:", answer_cols)

df_raw['review_content'] = df_raw[answer_cols].fillna('').agg(' '.join, axis=1).str.strip()
df_raw['review_content'] = df_raw['review_content'].str.replace(r'\s+', ' ', regex=True)

df_raw = df_raw[df_raw['review_content'].str.len() > 20].reset_index(drop=True)
print("After dropping empty reviews:", df_raw.shape)

Raw shape: (103439, 82)
Columns: ['answers/0', 'answers/1', 'answers/2', 'answers/3', 'answers_raw', 'date/published', 'date/submitted', 'date/updated', 'helpful', 'id', 'industry', 'location/country', 'location/primary', 'location/region', 'name', 'product/id', 'product/name', 'product/slug', 'role', 'score', 'segment', 'source/review', 'source/type', 'switched_from/products', 'switched_from/products/0/id', 'switched_from/products/0/name', 'switched_from/products/0/slug', 'switched_from/products/1/id', 'switched_from/products/1/name', 'switched_from/products/1/slug', 'switched_from/products/2/id', 'switched_from/products/2/name', 'switched_from/products/2/slug', 'switched_from/products/3/id', 'switched_from/products/3/name', 'switched_from/products/3/slug', 'switched_from/products/4/id', 'switched_from/products/4/name', 'switched_from/products/4/slug', 'switched_from/products/5/id', 'switched_from/products/5/name', 'switched_from/products/5/slug', 'switched_from/products/6/id', 'switc

# **Keyword labeling**

In [7]:
def label_intent(text):
    text_lower = text.lower()

    # High Purchase Intent — switched TO the product
    hpi_keywords = [
        'we chose', 'we selected', 'we went with', 'we decided on',
        'we purchased', 'we bought', 'we implemented', 'we adopted',
        'we migrated to', 'we switched to', 'we moved to',
        'after evaluating', 'after comparing', 'after a trial',
        'switched from', 'moved from', 'replaced', 'we use it instead'
    ]

    # Churn Intent — leaving or looking to leave
    churn_keywords = [
        'looking for alternatives', 'considering alternatives',
        'switching away', 'moving away', 'cancelling', 'cancel our',
        'not renewing', 'will not renew', 'looking to replace',
        'evaluating other options', 'too expensive', 'pricing is too high',
        'we left', 'we stopped using', 'we churned', 'looking elsewhere',
        'contract ends', 'when our contract', 'not worth the price'
    ]

    # Evaluation Intent — trialing or comparing
    eval_keywords = [
        'free trial', 'during our trial', 'evaluating', 'piloting',
        'testing out', 'we are comparing', 'comparing alternatives',
        'proof of concept', 'poc', 'demo', 'sandbox',
        'shortlisted', 'on our shortlist', 'considering it'
    ]

    # Advocacy Intent — recommending to others
    advocacy_keywords = [
        'i recommend', 'highly recommend', 'would recommend',
        'i suggest', 'tell others', 'suggest to anyone',
        'recommend it to', 'recommend this to', 'recommend to my',
        'would tell', 'tell my colleagues', 'suggest my team'
    ]

    for kw in hpi_keywords:
        if kw in text_lower:
            return 'High Purchase Intent'
    for kw in churn_keywords:
        if kw in text_lower:
            return 'Churn Intent'
    for kw in eval_keywords:
        if kw in text_lower:
            return 'Evaluation Intent'
    for kw in advocacy_keywords:
        if kw in text_lower:
            return 'Advocacy Intent'

    return 'Neutral'

df_raw['intent_label'] = df_raw['review_content'].apply(label_intent)

print("\nLabel distribution (raw, unbalanced):")
print(df_raw['intent_label'].value_counts())
print("\nPercentages:")
print(df_raw['intent_label'].value_counts(normalize=True).mul(100).round(1))


Label distribution (raw, unbalanced):
intent_label
Neutral                 96513
Advocacy Intent          3021
Evaluation Intent        1063
High Purchase Intent      742
Churn Intent              289
Name: count, dtype: int64

Percentages:
intent_label
Neutral                 95.0
Advocacy Intent          3.0
Evaluation Intent        1.0
High Purchase Intent     0.7
Churn Intent             0.3
Name: proportion, dtype: float64


In [8]:
total = len(df_raw)
labeled = df_raw[df_raw['intent_label'] != 'Neutral'].shape[0]
neutral = df_raw[df_raw['intent_label'] == 'Neutral'].shape[0]

print("=" * 55)
print("DATASET STATS — THE NUMBERS TO KNOW FOR YOUR DEFENSE")
print("=" * 55)
print(f"Total raw reviews:          {total:>10,}")
print(f"Labeled (non-Neutral):      {labeled:>10,}  ({labeled/total*100:.1f}%)")
print(f"Neutral (dropped/reduced):  {neutral:>10,}  ({neutral/total*100:.1f}%)")
print(f"Final balanced dataset:     {12154:>10,}  ({12154/total*100:.1f}% of raw)")
print("=" * 55)

DATASET STATS — THE NUMBERS TO KNOW FOR YOUR DEFENSE
Total raw reviews:             101,628
Labeled (non-Neutral):           5,115  (5.0%)
Neutral (dropped/reduced):      96,513  (95.0%)
Final balanced dataset:         12,154  (12.0% of raw)


# **LR and SVM on unbalanced data**

In [9]:
# Use all labeled data — no balancing, no cap
df_unbalanced = df_raw.copy()
df_unbalanced['clean_review'] = df_unbalanced['review_content']

X = df_unbalanced['clean_review']
y = df_unbalanced['intent_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vec.fit_transform(X_train)
X_test_tfidf = vec.transform(X_test)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print("=" * 55)
print("LOGISTIC REGRESSION — UNBALANCED DATA")
print("=" * 55)
print(f"Accuracy: {acc_lr:.2f}  |  Weighted F1: {f1_lr:.2f}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred_lr))

# SVM
svm = LinearSVC(max_iter=2000, random_state=42)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print("=" * 55)
print("SVM — UNBALANCED DATA")
print("=" * 55)
print(f"Accuracy: {acc_svm:.2f}  |  Weighted F1: {f1_svm:.2f}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred_svm))

LOGISTIC REGRESSION — UNBALANCED DATA
Accuracy: 0.97  |  Weighted F1: 0.96

Detailed Report:
                      precision    recall  f1-score   support

     Advocacy Intent       0.88      0.67      0.76       604
        Churn Intent       1.00      0.17      0.29        58
   Evaluation Intent       1.00      0.12      0.21       213
High Purchase Intent       0.54      0.05      0.09       148
             Neutral       0.97      1.00      0.98     19303

            accuracy                           0.97     20326
           macro avg       0.88      0.40      0.47     20326
        weighted avg       0.97      0.97      0.96     20326

SVM — UNBALANCED DATA
Accuracy: 0.98  |  Weighted F1: 0.98

Detailed Report:
                      precision    recall  f1-score   support

     Advocacy Intent       0.87      0.79      0.83       604
        Churn Intent       0.97      0.55      0.70        58
   Evaluation Intent       0.97      0.34      0.51       213
High Purchase Intent

In [10]:
print("=" * 65)
print("COMPARISON SUMMARY")
print("=" * 65)
print(f"{'Model':<25} {'Unbalanced Acc':>15} {'Balanced Acc':>15}")
print("-" * 65)
print(f"{'Logistic Regression':<25} {acc_lr:.2f}{'':>12} {'0.72':>12}")
print(f"{'SVM':<25} {acc_svm:.2f}{'':>12} {'0.83':>12}")
print("=" * 65)
print("\nNote: balanced results from main notebook (12,154 reviews)")
print("Unbalanced = full labeled dataset before 2,000 class cap")

COMPARISON SUMMARY
Model                      Unbalanced Acc    Balanced Acc
-----------------------------------------------------------------
Logistic Regression       0.97                     0.72
SVM                       0.98                     0.83

Note: balanced results from main notebook (12,154 reviews)
Unbalanced = full labeled dataset before 2,000 class cap
